In [1]:
from pyspark.sql import SparkSession

spark = SparkSession.builder \
    .appName("Sales ETL Pipeline") \
    .getOrCreate()

In [2]:
#Read CSV file into a DataFrame
df = spark.read.csv(
    "data/sales.csv",
    header=True,
    inferSchema=True
)

df.show()

+--------+--------+--------+--------+-----+
|Order_ID|Customer| Product|Quantity|Price|
+--------+--------+--------+--------+-----+
|     101|   Alice|  Laptop|       2|60000|
|     102|     Bob|   Mouse|       3|  700|
|     103| Charlie|Keyboard|       1| 1500|
|     104|   Alice| Monitor|       2|12000|
|     105|    NULL|  Laptop|       1|60000|
|     106|   David|   Mouse|    NULL|  700|
|     107|     Eva|Keyboard|       2| 1500|
|     107|     Eva|Keyboard|       2| 1500|
|     108|   Frank|  Laptop|       1|65000|
+--------+--------+--------+--------+-----+



In [3]:
#Check the schema of the DataFrame
df.printSchema()

root
 |-- Order_ID: integer (nullable = true)
 |-- Customer: string (nullable = true)
 |-- Product: string (nullable = true)
 |-- Quantity: integer (nullable = true)
 |-- Price: integer (nullable = true)



In [4]:
#Remove duplicate records from the DataFrame
df = df.dropDuplicates()
df.show()

+--------+--------+--------+--------+-----+
|Order_ID|Customer| Product|Quantity|Price|
+--------+--------+--------+--------+-----+
|     101|   Alice|  Laptop|       2|60000|
|     103| Charlie|Keyboard|       1| 1500|
|     102|     Bob|   Mouse|       3|  700|
|     108|   Frank|  Laptop|       1|65000|
|     104|   Alice| Monitor|       2|12000|
|     107|     Eva|Keyboard|       2| 1500|
|     106|   David|   Mouse|    NULL|  700|
|     105|    NULL|  Laptop|       1|60000|
+--------+--------+--------+--------+-----+



In [5]:
#Handling Missing Values
#Fill missing values in the 'quantity' column with 1
df = df.fillna({
    "Quantity":1
})

#Remove rows where 'customer' is null
df = df.na.drop(subset=["Customer"])

df.show()

+--------+--------+--------+--------+-----+
|Order_ID|Customer| Product|Quantity|Price|
+--------+--------+--------+--------+-----+
|     101|   Alice|  Laptop|       2|60000|
|     103| Charlie|Keyboard|       1| 1500|
|     102|     Bob|   Mouse|       3|  700|
|     108|   Frank|  Laptop|       1|65000|
|     104|   Alice| Monitor|       2|12000|
|     107|     Eva|Keyboard|       2| 1500|
|     106|   David|   Mouse|       1|  700|
+--------+--------+--------+--------+-----+



In [6]:
#Create a new column 
from pyspark.sql.functions import col
#Create total sales 
df = df.withColumn(
    "Total_Sales",
    col("Quantity") * col("Price")
)
df.show()

+--------+--------+--------+--------+-----+-----------+
|Order_ID|Customer| Product|Quantity|Price|Total_Sales|
+--------+--------+--------+--------+-----+-----------+
|     101|   Alice|  Laptop|       2|60000|     120000|
|     103| Charlie|Keyboard|       1| 1500|       1500|
|     102|     Bob|   Mouse|       3|  700|       2100|
|     108|   Frank|  Laptop|       1|65000|      65000|
|     104|   Alice| Monitor|       2|12000|      24000|
|     107|     Eva|Keyboard|       2| 1500|       3000|
|     106|   David|   Mouse|       1|  700|        700|
+--------+--------+--------+--------+-----+-----------+



In [7]:
#Filter Expensive Products
filtered_df = df.filter(col("Price") > 1000)

filtered_df.show()

+--------+--------+--------+--------+-----+-----------+
|Order_ID|Customer| Product|Quantity|Price|Total_Sales|
+--------+--------+--------+--------+-----+-----------+
|     101|   Alice|  Laptop|       2|60000|     120000|
|     103| Charlie|Keyboard|       1| 1500|       1500|
|     108|   Frank|  Laptop|       1|65000|      65000|
|     104|   Alice| Monitor|       2|12000|      24000|
|     107|     Eva|Keyboard|       2| 1500|       3000|
+--------+--------+--------+--------+-----+-----------+



In [8]:
#Sort data
df.orderBy(col("Total_Sales").desc()).show()

+--------+--------+--------+--------+-----+-----------+
|Order_ID|Customer| Product|Quantity|Price|Total_Sales|
+--------+--------+--------+--------+-----+-----------+
|     101|   Alice|  Laptop|       2|60000|     120000|
|     108|   Frank|  Laptop|       1|65000|      65000|
|     104|   Alice| Monitor|       2|12000|      24000|
|     107|     Eva|Keyboard|       2| 1500|       3000|
|     102|     Bob|   Mouse|       3|  700|       2100|
|     103| Charlie|Keyboard|       1| 1500|       1500|
|     106|   David|   Mouse|       1|  700|        700|
+--------+--------+--------+--------+-----+-----------+



In [9]:
#Group By : Find total sales per customer
from pyspark.sql.functions import sum

customer_sales = df.groupBy("Customer") \
                   .agg(sum("Total_Sales").alias("Total"))

customer_sales.show()

+--------+------+
|Customer| Total|
+--------+------+
|     Eva|  3000|
| Charlie|  1500|
|     Bob|  2100|
|   Alice|144000|
|   David|   700|
|   Frank| 65000|
+--------+------+



In [10]:
import os
print(os.getcwd())

c:\Users\ASUS\OneDrive\Desktop\PySpark_ETL_Project


In [11]:
df.show()
df.printSchema()

+--------+--------+--------+--------+-----+-----------+
|Order_ID|Customer| Product|Quantity|Price|Total_Sales|
+--------+--------+--------+--------+-----+-----------+
|     101|   Alice|  Laptop|       2|60000|     120000|
|     103| Charlie|Keyboard|       1| 1500|       1500|
|     102|     Bob|   Mouse|       3|  700|       2100|
|     108|   Frank|  Laptop|       1|65000|      65000|
|     104|   Alice| Monitor|       2|12000|      24000|
|     107|     Eva|Keyboard|       2| 1500|       3000|
|     106|   David|   Mouse|       1|  700|        700|
+--------+--------+--------+--------+-----+-----------+

root
 |-- Order_ID: integer (nullable = true)
 |-- Customer: string (nullable = true)
 |-- Product: string (nullable = true)
 |-- Quantity: integer (nullable = false)
 |-- Price: integer (nullable = true)
 |-- Total_Sales: integer (nullable = true)



In [12]:
import os
print(os.environ.get('HADOOP_HOME'))

C:\Users\ASUS\Hadoop


In [13]:
#Write Parquet
df.write.mode("overwrite").parquet("Output/sales_parquet")

In [15]:
#Read Parquet
parquet_df = spark.read.parquet(
    "output/sales_parquet"
)

parquet_df.show()

+--------+--------+--------+--------+-----+-----------+
|Order_ID|Customer| Product|Quantity|Price|Total_Sales|
+--------+--------+--------+--------+-----+-----------+
|     101|   Alice|  Laptop|       2|60000|     120000|
|     103| Charlie|Keyboard|       1| 1500|       1500|
|     102|     Bob|   Mouse|       3|  700|       2100|
|     108|   Frank|  Laptop|       1|65000|      65000|
|     104|   Alice| Monitor|       2|12000|      24000|
|     107|     Eva|Keyboard|       2| 1500|       3000|
|     106|   David|   Mouse|       1|  700|        700|
+--------+--------+--------+--------+-----+-----------+

